# Adult MALLM-GAN Reference-Synthetic Pipeline

This notebook runs direction 2: use existing generated data as the reference source instead of raw real training samples.

Inputs: `gen/adult/{sample_size}/df_{seed}.csv`

Outputs: `gen/adult_reference_synthetic/{sample_size}/df_{seed}.csv`

For each sample size, this notebook reruns the model once per existing generated seed file. For example, `gen/adult/100/df_3.csv` becomes the reference source for the new `gen/adult_reference_synthetic/100/df_3.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI

from eval_utils import data_profiling, cate_info
from model_glm_reference_synthetic import (
    ReferenceSyntheticGAN,
    load_reference_synthetic_data,
)

np.random.seed(42)

## Schema

In [2]:
x_cols = [
    "age",
    "workclass",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
]
y_col = "Income"
cate_var = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
    "Income",
]
bool_var = []
cols = x_cols + [y_col]
num_var = list((set(cols) - set(cate_var)) - set(bool_var))

data_desc = (
    "The dataset includes social-economic and demographic variables, "
    "with a binary income label indicating whether income is higher than 50K. "
    "This run uses existing generated reference data instead of raw real rows."
)

params = {
    "max_depth": 3,
    "eta": 0.3,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
}

## LLM Client

Set your API key in the environment before running the notebook. For PowerShell:

```powershell
$env:OPENAI_API_KEY="your_api_key"
$env:OPENAI_BASE_URL="https://api.deepseek.com"
```

If you use OpenAI's default endpoint, omit `OPENAI_BASE_URL`.

In [3]:
import os

os.environ["OPENAI_API_KEY"] = "sk-fe430666ae7444b8932bb8165c70fda8"
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"

api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")

if not api_key:
    raise ValueError("Please set OPENAI_API_KEY before running this notebook.")

client_kwargs = {"api_key": api_key}
if base_url:
    client_kwargs["base_url"] = base_url

gen_client = OpenAI(**client_kwargs)
opt_client = OpenAI(**client_kwargs)

gen_model_nm = os.environ.get("GEN_MODEL", "deepseek-chat")
opt_model_nm = os.environ.get("OPT_MODEL", "deepseek-chat")

## Run Configuration

For a quick smoke test, use `num_samples = [100]`, `epochs = {100: 1}`, and `reference_seed_lst = [0]`.

In [4]:
reference_root = "gen/adult"
output_root = "gen/adult_reference_synthetic"
log_root = "log/reference_synthetic"

# num_samples = [100, 200, 400, 800]
# epochs = {100: 5, 200: 4, 400: 3, 800: 2}
num_samples = [800]
epochs = {800: 2}
batch_size = 50
reference_seed_lst = list(range(5))

os.makedirs(output_root, exist_ok=True)
os.makedirs(log_root, exist_ok=True)

## Generate From Existing Generated Reference Data

In [5]:
res = {}
model_dict = {}

for num in num_samples:
    output_dir = f"{output_root}/{num}"
    os.makedirs(output_dir, exist_ok=True)

    res[str(num)] = {}
    model_dict[str(num)] = {}

    for reference_seed in reference_seed_lst:
        reference_file = f"{reference_root}/{num}/df_{reference_seed}.csv"
        reference_data = load_reference_synthetic_data(reference_file, cols)

        metadata = data_profiling(reference_data, cate_var, bool_var, cols)
        cate_desc = cate_info(reference_data, cate_var)

        log_file = f"{log_root}/adult_reference_synthetic_{num}_seed_{reference_seed}.txt"
        epoch = epochs[num]

        magan = ReferenceSyntheticGAN(
            gen_client,
            opt_client,
            gen_model_nm,
            opt_model_nm,
            params,
            reference_data,
            cols,
            y_col,
            num_var,
            metadata,
            cate_desc,
            data_desc,
            log_file,
            opt_temperature=1,
            real_samples_num=5,
        )

        magan._run(batch_size, epoch)
        magan.run_with_fixed_discriminator(0, epoch, batch_size, 1)
        model_dict[str(num)][str(reference_seed)] = magan

        mallm_syn_df = magan.gen_without_optimization(num_folds=1)
        output_file = f"{output_dir}/df_{reference_seed}.csv"
        mallm_syn_df.to_csv(output_file, index=False)

        res[str(num)][str(reference_seed)] = {
            "reference_file": reference_file,
            "output_file": output_file,
            "rows": len(mallm_syn_df),
        }
        print(f"Reference {reference_file} -> wrote {output_file}: {len(mallm_syn_df)} rows")

[bnlearn] >Warning: Computing DAG with 14 nodes can take a very long time!
[bnlearn] >Computing best DAG using [hc]
[bnlearn] >Set scoring type at [bic]


[bnlearn] >Compute structure scores for model comparison (higher is better).


INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx

Reference gen/adult/800/df_0.csv -> wrote gen/adult_reference_synthetic/800/df_0.csv: 800 rows
[bnlearn] >Warning: Computing DAG with 14 nodes can take a very long time!
[bnlearn] >Computing best DAG using [hc]
[bnlearn] >Set scoring type at [bic]


INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


...

Reference gen/adult/800/df_3.csv -> wrote gen/adult_reference_synthetic/800/df_3.csv: 800 rows
[bnlearn] >Warning: Computing DAG with 14 nodes can take a very long time!
[bnlearn] >Computing best DAG using [hc]
[bnlearn] >Set scoring type at [bic]


INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


[bnlearn] >Compute structure scores for model comparison (higher is better).


INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx

Reference gen/adult/800/df_4.csv -> wrote gen/adult_reference_synthetic/800/df_4.csv: 800 rows


## Save Run Summary

In [6]:
summary_rows = []
for sample_size, seed_info in res.items():
    for seed, info in seed_info.items():
        summary_rows.append(
            {
                "sample_size": int(sample_size),
                "seed": int(seed),
                "reference_file": info["reference_file"],
                "output_file": info["output_file"],
                "rows": info["rows"],
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_path = f"{output_root}/run_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df

,sample_size,seed,reference_file,output_file,rows
0,800,0,gen/adult/800/df_0.csv,gen/adult_reference_synthetic/800/df_0.csv,800
1,800,1,gen/adult/800/df_1.csv,gen/adult_reference_synthetic/800/df_1.csv,800
2,800,2,gen/adult/800/df_2.csv,gen/adult_reference_synthetic/800/df_2.csv,800
3,800,3,gen/adult/800/df_3.csv,gen/adult_reference_synthetic/800/df_3.csv,800
4,800,4,gen/adult/800/df_4.csv,gen/adult_reference_synthetic/800/df_4.csv,800
